# 02 · Counterfactual reasoning with `do()`

The core causal capability: given a factual scenario, estimate the outcome under a counterfactual `do()` intervention — 'what if I had doubled the budget?' 'what if I had switched to a mega-tier KOL?'

This notebook uses the LightGBM baseline for speed (sub-millisecond). The research-grade Causal Transformer world model exposes the same API but adds per-arm counterfactual heads trained with representation-balancing regularization — see `oransim.world_model.CausalTransformerWorldModel`.

In [ ]:
import pickle
from pathlib import Path

import lightgbm as lgb
import numpy as np

with open(Path('../data/models/world_model_demo.pkl'), 'rb') as f:
    blob = pickle.load(f)
niches = blob['config']['niches']
tiers = blob['config']['kol_tiers']
boosters = {
    kpi: {float(q): lgb.Booster(model_str=s) for q, s in per_q.items()}
    for kpi, per_q in blob['boosters'].items()
}

def featurize(scn):
    return np.asarray([[
        0.0,
        float(niches.index(scn['niche'])),
        scn['budget'],
        float(scn['budget_bucket']),
        float(tiers.index(scn['kol_tier'])),
        float(scn['kol_fan_count']),
        scn['kol_engagement_rate'],
    ]], dtype=np.float32)

def predict_p50(scn):
    x = featurize(scn)
    return {kpi: float(boosters[kpi][0.50].predict(x)[0])
            for kpi in ('impressions','clicks','conversions','revenue')}

In [ ]:
# Factual scenario
factual = {
    'niche': 'beauty', 'budget': 40_000.0, 'budget_bucket': 1,
    'kol_tier': 'micro', 'kol_fan_count': 60_000, 'kol_engagement_rate': 0.035,
}

# Counterfactual: do(budget ← 2x)
cf_budget = dict(factual, budget=80_000.0, budget_bucket=2)

# Counterfactual: do(kol_tier ← mid + 5x fans)
cf_tier = dict(factual, kol_tier='mid', kol_fan_count=300_000)

pred_factual = predict_p50(factual)
pred_cf_budget = predict_p50(cf_budget)
pred_cf_tier = predict_p50(cf_tier)

print(f"{'':<14} {'factual':>12} {'do(2x budget)':>16} {'Δ%':>8} | {'do(→ mid KOL)':>16} {'Δ%':>8}")
print('-'*82)
for kpi in ('impressions','clicks','conversions','revenue'):
    f, a, b = pred_factual[kpi], pred_cf_budget[kpi], pred_cf_tier[kpi]
    da = 100.0 * (a-f) / (abs(f) + 1e-9)
    db = 100.0 * (b-f) / (abs(f) + 1e-9)
    print(f"{kpi:<14} {f:>12.1f} {a:>16.1f} {da:>+7.1f}% | {b:>16.1f} {db:>+7.1f}%")

Note the **Hill saturation** effect — doubling budget does *not* double impressions. The world model learned this pattern from the synthetic corpus where it's baked into the ground-truth generator.

## Going further

With the `[ml]` extra and PyTorch installed, you can use the research-grade Causal Transformer world model which exposes per-arm counterfactual heads trained with HSIC representation balancing (BCAUSS). Code:

```python
from oransim.world_model import get_world_model
wm = get_world_model('causal_transformer')
factual_pred = wm.predict(features)
cf_pred = wm.counterfactual(features, arm_idx=2)
```

See `oransim/world_model/transformer.py` for architecture + references.